# Multi-follower Stackelberg experiment

Each entry in `scenarios`, `lower_nets`, and `mg_buses` describes the same follower. Every follower receives an independent 24-hour price trajectory when more than one follower is configured. Run the notebook from top to bottom in this order: neural-surrogate solver (NN), zero-order solver (CMA-ES), and residual-correction trust-region algorithm.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower.networks as pn
 
from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.nn_solver import (
    NNSolverExperimentConfig,
    run_nn_solver_experiment,
)
from stackelberg.trust_region import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)
from stackelberg.zero_order import (
    ZeroOrderExperimentConfig,
    run_zero_order_experiment,
)

## Networks and follower mapping

The order is important: follower `i` uses `scenarios[i]`, `lower_nets[i]`, connects at `mg_buses[i]`, and loads `L2O_<follower_id>.pth`.

In [2]:
T = 24
upper_net = create_case33bw_20kv()

# The three lists use the same follower order.
scenarios = ["SB1", "SB2", "BG1", "BG2", "IP1",]
# SB uses pn.create_cigre_network_lv()
# BG uses create_cigre_residential_network()
lower_nets = [
    pn.create_cigre_network_lv(),
    pn.create_cigre_network_lv(),
    create_cigre_residential_network(),
    create_cigre_residential_network(),
    create_industrial_park_network(),
]
mg_buses = [17, 32, 15, 21, 1]

rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(1)
# Shared initial prices make the three methods directly comparable.
x0 = np.full((len(scenarios), T), 0.75) # the center point 9.12
# x0 = np.full((len(scenarios), T), 0.0) # the lower bound  9.13
# x0 = np.full((len(scenarios), T), 1.5) # the upper bound 9.13
# x0 = rng1.random((len(scenarios), T)) * 1.5 # the random starting point 9.14
# x0 = rng2.random((len(scenarios), T)) * 1.5 # the random starting point 9.14
x0

array([[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75]])

## 1. Neural-surrogate solver (NN) or saved warm start

Set `RUN_NN_SOLVER=True` to run and save a new NN solution. Leave it `False` to read the last recorded price vector from the newest `results/NN_solver_history_*.csv` without rerunning the solver.

In [4]:
RUN_NN_SOLVER = True

nn_config = NNSolverExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    device="cpu",
    x0=x0,
    steps=10000,
    lr=1e-2,
    save_every=200,
    history_path="results/NN_solver_history.csv",
    evaluate_true=True,
    save_results=True,
)

if RUN_NN_SOLVER:
    nn_result = run_nn_solver_experiment(nn_config)
    nn_prices = np.asarray(nn_result["optimal pi"]).reshape(len(scenarios), T)
    warm_start_point = nn_prices.copy()
    print("Ran NN solver and saved history to:", nn_result.get("history_path"))
else:
    results_dir = Path("results")
    history_candidates = list(results_dir.glob("NN_solver_history_*.csv"))
    if not history_candidates:
        fallback_history = results_dir / "NN_solver_history.csv"
        if fallback_history.is_file():
            history_candidates = [fallback_history]
    if not history_candidates:
        raise FileNotFoundError(
            "No NN solver history found in results/. Expected NN_solver_history_*.csv."
        )

    latest_nn_history_path = max(
        history_candidates,
        key=lambda path: path.stat().st_mtime_ns,
    )
    nn_history = pd.read_csv(latest_nn_history_path)
    if nn_history.empty:
        raise ValueError(f"NN solver history is empty: {latest_nn_history_path}")

    pi_columns = sorted(
        (column for column in nn_history.columns if column.startswith("pi_")),
        key=lambda column: int(column.removeprefix("pi_")),
    )
    expected_price_count = len(scenarios) * T
    if len(pi_columns) != expected_price_count:
        raise ValueError(
            f"Expected {expected_price_count} price columns, found {len(pi_columns)} "
            f"in {latest_nn_history_path}."
        )

    last_nn_row = nn_history.sort_values("iteration").iloc[-1]
    nn_prices = last_nn_row[pi_columns].to_numpy(dtype=float).reshape(
        len(scenarios), T
    )
    warm_start_point = nn_prices.copy()
    nn_result = {
        "optimal pi": nn_prices,
        "true_obj": float(last_nn_row["true_loss"])
        if "true_loss" in last_nn_row.index
        else np.nan,
        "runtime_seconds": np.nan,
        "history_path": str(latest_nn_history_path),
    }
    print("Loaded NN warm start from:", latest_nn_history_path)
    print("Saved iteration:", int(last_nn_row["iteration"]))

print("NN true objective:", nn_result["true_obj"])
print("Runtime seconds:", nn_result["runtime_seconds"])
display(pd.DataFrame(nn_prices, index=scenarios))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

[0] loss=5821.5625, obj=-123.5207, eq=5.9380, soc=0.0647, bound=0.0646, true_loss=-0.1048
Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copy

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.970181,0.972706,0.975164,1.000694,0.959616,0.969702,1.160907,1.497205,1.497323,0.998421,...,1.047077,1.499775,1.496677,1.499511,1.337253,1.495676,1.489284,1.496565,1.093230,0.966810
SB2,0.984327,1.489236,0.971531,0.980285,0.946286,0.950465,0.998462,0.976595,1.498717,1.498230,...,1.016355,1.498472,1.497773,1.498430,1.498577,1.499142,1.497796,1.451597,1.440844,1.184781
BG1,1.499816,1.499887,1.499895,1.499872,1.499943,1.499931,1.499945,1.499928,1.499881,1.499913,...,1.499950,1.499931,1.499941,1.499935,1.499918,1.499891,1.499909,1.499468,1.499928,1.499742
BG2,1.499647,1.499705,1.499627,1.499876,1.499915,1.499792,1.499918,1.499930,1.499858,1.499912,...,1.499942,1.499920,1.499952,1.499877,1.499927,1.499919,1.499869,1.499671,1.499878,1.499435
IP1,1.496455,1.492518,1.494619,1.495886,1.491641,1.457786,1.486227,1.497947,1.498367,1.497680,...,1.498705,1.498223,1.498699,1.496387,1.490192,1.497343,1.496753,1.495072,1.498292,1.497317


In [6]:
assert warm_start_point.shape == (len(scenarios), T)
warm_start_point

array([[0.9510933 , 0.9677322 , 1.4008077 , 0.99608564, 0.9815737 ,
        0.9863839 , 1.1597009 , 1.4986484 , 1.3798103 , 1.0001589 ,
        1.0024807 , 1.004489  , 1.0167059 , 1.2428774 , 1.0509703 ,
        1.4998065 , 1.4972873 , 1.4994323 , 1.3076943 , 1.4977436 ,
        1.4981138 , 1.4071102 , 1.0876321 , 1.3673854 ],
       [0.9850111 , 1.4881468 , 0.9716479 , 0.9803585 , 0.9486345 ,
        0.95089173, 0.9976368 , 0.9765725 , 1.4987451 , 1.4982271 ,
        1.3413062 , 1.0087876 , 1.4982159 , 1.3030633 , 1.020771  ,
        1.4983419 , 1.4976981 , 1.498402  , 1.4985148 , 1.49913   ,
        1.4977973 , 1.454737  , 1.4409869 , 1.1853588 ],
       [1.4998174 , 1.4998873 , 1.4998938 , 1.4998713 , 1.4999433 ,
        1.4999311 , 1.4999446 , 1.4999282 , 1.4998808 , 1.4999135 ,
        1.4999182 , 1.4999456 , 1.4999204 , 1.4999328 , 1.4999503 ,
        1.4999306 , 1.4999406 , 1.4999352 , 1.4999185 , 1.4998922 ,
        1.4999099 , 1.4994696 , 1.4999292 , 1.499718  ],
       [1.499

## 2. Zero-order solver (CMA-ES) or saved result

Set `RUN_CMA_ES=True` to run and save a new CMA-ES solution. Leave it `False` to load the newest `results/zero_solver_result_*.json` and its matching interaction history.

In [7]:
RUN_CMA_ES = True

zero_config = ZeroOrderExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    pi_0=warm_start_point,
    maxiter=500,
    cma_verbose=1,
    # Per-coordinate initial standard deviation. Use None for the legacy
    # (pi_max - pi_min) / 4 value; 0.05-0.075 favors local warm-start search.
    cma_sigma0= 0.05,
    # Applied to every individual Gurobi lower-level solve.
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    save_results=True,
    output_dir="results",
)

if RUN_CMA_ES:
    zero_result = run_zero_order_experiment(zero_config)
    print("Ran CMA-ES and saved:", zero_result.get("saved_paths"))
else:
    results_dir = Path(zero_config.output_dir)
    result_candidates = list(results_dir.glob("zero_solver_result_*.json"))
    if not result_candidates:
        raise FileNotFoundError(
            "No saved CMA-ES result found in results/. "
            "Expected zero_solver_result_*.json."
        )

    latest_zero_result_path = max(
        result_candidates,
        key=lambda path: path.stat().st_mtime_ns,
    )
    with latest_zero_result_path.open("r", encoding="utf-8") as file:
        saved_zero_result = json.load(file)

    saved_scenarios = [str(value) for value in saved_zero_result["scenarios"]]
    if saved_scenarios != list(scenarios):
        raise ValueError(
            f"Saved CMA-ES scenarios {saved_scenarios} do not match {scenarios}."
        )
    saved_prices = np.asarray(saved_zero_result["optimal_pi"], dtype=float)
    if saved_prices.size != len(scenarios) * T:
        raise ValueError(
            f"Expected {len(scenarios) * T} saved CMA-ES prices, "
            f"found {saved_prices.size}."
        )

    result_timestamp = latest_zero_result_path.stem.removeprefix(
        "zero_solver_result_"
    )
    zero_history_path = results_dir / f"zero_solver_history_{result_timestamp}.csv"
    zero_solution_path = results_dir / f"zero_solver_solution_{result_timestamp}.csv"
    if not zero_history_path.is_file():
        raise FileNotFoundError(
            f"Matching CMA-ES interaction history not found: {zero_history_path}"
        )

    zero_result = {
        "optimal pi": saved_prices.reshape(len(scenarios), T),
        "optimal value": float(saved_zero_result["optimal_value"]),
        "p_mg": saved_zero_result.get("p_mg"),
        "runtime_seconds": float(saved_zero_result["runtime_seconds"]),
        "n_interactions": int(saved_zero_result["n_interactions"]),
        "interaction_history": pd.read_csv(zero_history_path),
        "scenarios": saved_scenarios,
        "mg_buses": saved_zero_result.get("mg_buses"),
        "saved_paths": {
            "json": str(latest_zero_result_path),
            "solution_csv": str(zero_solution_path),
            "history_csv": str(zero_history_path),
        },
    }
    print("Loaded CMA-ES result from:", latest_zero_result_path)

zero_prices = np.asarray(zero_result["optimal pi"]).reshape(len(scenarios), T)
print("CMA-ES optimal value:", zero_result["optimal value"])
print("Runtime seconds:", zero_result["runtime_seconds"])
print("Composite high-fidelity queries:", zero_result["n_interactions"])
print(
    "Individual follower solves:",
    zero_result["n_interactions"] * len(scenarios),
)
print("Saved paths:", zero_result.get("saved_paths"))
display(pd.DataFrame(zero_prices, index=scenarios))

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

(9_w,18)-aCMA-ES (mu_w=5.4,w_1=30%) in dimension 120 (seed=42, Wed Sep 16 22:52:10 2026)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     18 2.041955561596138e+02 1.0e+00 4.82e-02  5e-02  5e-02 0:16.4
    2     36 1.463448070454100e+02 1.0e+00 4.69e-02  5e-02  5e-02 0:33.9
    3     54 -9.184041770483636e+01 1.0e+00 4.59e-0

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,1.000764,0.951453,1.226839,0.998371,0.820113,0.989783,1.000148,1.450962,1.278036,1.140763,...,1.275824,1.322632,1.499565,1.496771,1.316467,1.432289,1.463830,1.450104,1.167794,1.494034
SB2,0.999276,1.365591,0.958285,0.999377,1.000636,0.994044,0.999899,0.996446,1.496174,1.497980,...,0.932808,1.364776,1.445021,1.489042,1.477482,1.477643,1.499797,1.443169,1.353464,1.000439
BG1,1.426329,1.459714,1.497992,1.453508,1.498749,1.483454,1.499785,1.382570,1.458821,1.498793,...,1.493754,1.492974,1.456624,1.489156,1.499308,1.479360,1.492881,1.498874,1.471437,1.483389
BG2,1.490314,1.470965,1.481176,1.499993,1.485777,1.490048,1.498863,1.483601,1.499240,1.485613,...,1.497914,1.472783,1.486541,1.457370,1.467197,1.498671,1.499851,1.494503,1.465964,1.488074
IP1,1.499267,1.499905,1.499639,1.499920,1.499648,1.498439,1.499755,1.499130,1.498837,1.499556,...,1.499990,1.499270,1.499539,1.498863,1.499055,1.499560,1.499698,1.498917,1.499999,1.499812


In [8]:
baseline_summary = pd.DataFrame(
    [
        {
            "method": "NN solver",
            "true_objective": nn_result["true_obj"],
            "runtime_seconds": nn_result["runtime_seconds"],
        },
        {
            "method": "CMA-ES",
            "true_objective": zero_result["optimal value"],
            "runtime_seconds": zero_result["runtime_seconds"],
        },
    ]
).set_index("method")
display(baseline_summary)

,true_objective,runtime_seconds
method,,
NN solver,286.016949,2207.803818
CMA-ES,-99.850905,15205.432831


## 3. Trust-region algorithm

Run the residual-correction trust-region method last. The first leader solve starts from `x0` and uses `steps_init` on the global surrogate only. After that, each iteration samples true follower responses, fits a local residual, and re-solves with `steps_warm`. Sampled high-fidelity responses are saved as `data/X_{SB,BG,IP}_tr.npy` so they can retrain the NN solver without overwriting the original `X_*.npy` files. For a smoke test, reduce `K_max`, `n_samples`, `steps_init`, and `steps_warm`.

In [9]:
RESIDUAL_MODEL_TYPE = "rbf"  # Use "linear" for an affine residual.
RBF_KERNEL = "cubic"
RBF_SMOOTHING = 1e-3
RBF_EPSILON = None
RESIDUAL_UPDATE_MODE = "accumulate"  # Use "accumulate" for the previous behavior.

tr_config = TrustRegionExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    device="cpu",
    x0=warm_start_point,
    # Applied to every individual Gurobi lower-level solve.
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 60,
        "sampling_method": "lhs",
        "residual_update_mode": RESIDUAL_UPDATE_MODE,
        "rbf_geometry_condition_max": 1e12,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 2000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": True,
        },
    },
    residual_model_params={
        "model_type": RESIDUAL_MODEL_TYPE,
        "rbf_kernel": RBF_KERNEL,
        "rbf_smoothing": RBF_SMOOTHING,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius",
        "rbf_epsilon": RBF_EPSILON,
    },
    save_results=True,
    output_dir="results",
    # False: use local TR samples for fitting, but do not retain or export them.
    save_true_responses=False,
)

tr_result = run_trust_region_experiment(tr_config)
tr_result.get("saved_paths")


Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Cardinal Optimizer v8.0.6. Build date Aug  7 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Using global surrogate to solve initial point...
Initial surrogate solve used steps_init=0. Time taken to solve initial point 0.0721 seconds.
Iteration 0 | R=0.0500 
Building 60-point local designs for each of 5 followers (center + coordinate star + LHS) within R_tr=0.0500...
Fitting follower-local residual models with sample counts [60, 60, 60, 60, 60]...

RuntimeError: COPT failed with status 2

## Optional: retrain the NN solver on original + trust-region samples

Trust-region high-fidelity queries are stored only when `save_true_responses=True`. They are kept separately from the original training data:

- original: `data/X_SB.npy`, `data/Y_SB.npy` (and the same pattern for BG, IP)
- trust region: `data/X_SB_tr.npy`, `data/Y_SB_tr.npy` (and the same pattern for BG, IP)

To retrain, first set `save_true_responses=True` in `tr_config` and run the trust-region cell. Then set `RETRAIN_NN_FROM_TR_SAMPLES=True` to concatenate both sets (for example `X_BG.npy` + `X_BG_tr.npy`), train `L2O_{scenario}_tr.pth`, and re-solve with the new maps. The original `L2O_{scenario}.pth` checkpoints are left unchanged. The current experiment leaves both switches `False`.

In [ ]:
from L2Otraining import TrainingConfig, train

RETRAIN_NN_FROM_TR_SAMPLES = False

tr_nn_result = None
if RETRAIN_NN_FROM_TR_SAMPLES:
    for scenario in sorted(set(scenarios)):
        train(
            scenario,
            config=TrainingConfig(scenario=scenario, epochs=500, seed=42),
            data_tag="tr",
            include_original=True,
        )
    tr_nn_config = NNSolverExperimentConfig(
        upper_net=upper_net,
        lower_nets=lower_nets,
        mg_buses=mg_buses,
        scenarios=scenarios,
        T=T,
        seed=42,
        device="cpu",
        x0=x0,
        steps=5000,
        lr=1e-2,
        save_every=200,
        evaluate_true=True,
        save_results=True,
        model_tag="tr",
        history_path="results/NN_solver_tr_history.csv",
    )
    tr_nn_result = run_nn_solver_experiment(tr_nn_config)
    print("Retrained NN true objective:", tr_nn_result["true_obj"])
else:
    saved = (tr_result.get("saved_paths") or {}).get("true_responses", {})
    if saved:
        print("Trust-region samples ready for later NN retraining:")
        for scenario, paths in saved.items():
            print(f"  {scenario}: {paths['X']} , {paths['Y']}")
    else:
        print("No trust-region sample files were saved.")

## Compare results

Inspect all three solutions after the NN solver, CMA-ES, and trust-region cells have completed.

In [ ]:
tr_prices = np.asarray(tr_result["x_opt"]).reshape(len(scenarios), T)
history = pd.DataFrame(tr_result["history"])
last_iteration = tr_result["history"][-1]
tr_objective = (
    last_iteration["F_current"]
    if last_iteration["decision"] == "reject"
    else last_iteration["F_candidate"]
)
tr_sampling_queries = tr_result["n_iter"] * (tr_config.algo_params["n_samples"] + 1)

comparison = pd.DataFrame(
    [
        {
            "method": "NN solver",
            "true_objective": nn_result["true_obj"],
            "runtime_seconds": nn_result["runtime_seconds"],
            "composite_queries": np.nan,
            "individual_follower_solves": np.nan,
        },
        {
            "method": "CMA-ES",
            "true_objective": zero_result["optimal value"],
            "runtime_seconds": zero_result["runtime_seconds"],
            "composite_queries": zero_result["n_interactions"],
            "individual_follower_solves": zero_result["n_interactions"] * len(scenarios),
        },
        {
            "method": "Trust region",
            "true_objective": tr_objective,
            "runtime_seconds": tr_result["elapsed_seconds"],
            "composite_queries": tr_sampling_queries,
            "individual_follower_solves": tr_sampling_queries * len(scenarios),
        },
    ]
).set_index("method")

print("Trust region converged:", tr_result["converged"])
print("Trust region iterations:", tr_result["n_iter"])
print("Saved paths:", tr_result.get("saved_paths"))
display(comparison)
display(pd.concat(
    {
        "NN solver": pd.DataFrame(nn_prices, index=scenarios),
        "CMA-ES": pd.DataFrame(zero_prices, index=scenarios),
        "Trust region": pd.DataFrame(tr_prices, index=scenarios),
    },
    names=["method", "scenario"],
))
display(history[[
    "k", "F_current", "F_candidate", "rho", "decision", "R",
    "max_rbf_condition", "geometry_resample_attempts",
]].tail())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
history.plot(x="k", y=["F_current", "F_candidate"], ax=axes[0], title="Objective")
history.plot(x="k", y="rho", ax=axes[1], title="Trust ratio")
history.plot(x="k", y="R", ax=axes[2], title="Trust-region radius")
plt.tight_layout()

In [ ]:
# Compare methods using composite high-fidelity query accounting.
# CMA-ES n_interactions is the number of composite full-price queries.
zero_plot_df = zero_result["interaction_history"].copy()
zero_plot_df = zero_plot_df.sort_values("interaction").reset_index(drop=True)

tr_plot_df = history.copy()
tr_plot_df["accepted"] = tr_plot_df["decision"].str.startswith("accept")
tr_plot_df["cumulative_composite_queries"] = (
    (tr_plot_df["k"] + 1) * (tr_config.algo_params["n_samples"] + 1)
)

# Use the NN true loss as the query-zero incumbent for both warm-started methods.
# Any worse CMA-ES/TR objective is clipped to this baseline in the plot.
nn_true_loss = float(nn_result["true_obj"])
common_initial_objective = nn_true_loss

zero_plot_df = pd.concat(
    [
        # pd.DataFrame(
        #     {
        #         "interaction": [0],
        #         "upper_objective": [common_initial_objective],
        #     }
        # ),
        zero_plot_df[["interaction", "upper_objective"]],
    ],
    ignore_index=True,
)
# zero_plot_df["best_so_far_loss"] = np.minimum(
#     zero_plot_df["upper_objective"].cummin(),
#     nn_true_loss,
# )
zero_plot_df["best_so_far_loss"] = zero_plot_df["upper_objective"].cummin()

best_accepted = common_initial_objective
tr_best_values = []
for _, row in tr_plot_df.iterrows():
    if row["accepted"]:
        best_accepted = min(best_accepted, float(row["F_candidate"]))
    tr_best_values.append(best_accepted)
# tr_plot_df["best_true_objective"] = np.minimum(
#     tr_best_values,
#     nn_true_loss,
# )
tr_plot_df["best_true_objective"] = tr_best_values

tr_plot_df = pd.concat(
    [
        # pd.DataFrame(
        #     {
        #         "cumulative_composite_queries": [0],
        #         "best_true_objective": [common_initial_objective],
        #     }
        # ),
        tr_plot_df[[
            "cumulative_composite_queries",
            "best_true_objective",
        ]],
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    zero_plot_df["interaction"],
    zero_plot_df["best_so_far_loss"],
    linewidth=2.5,
    label="CMA-ES best-so-far",
)
ax.plot(
    tr_plot_df["cumulative_composite_queries"],
    tr_plot_df["best_true_objective"],
    linewidth=2.5,
    label="Trust region best accepted",
)
ax.axhline(
    nn_true_loss,
    color="tab:purple",
    linestyle="--",
    linewidth=2.5,
    label=f"NN solver true loss = {nn_true_loss:.3f}",
)
ax.scatter(
    [0],
    [common_initial_objective],
    color="black",
    s=70,
    zorder=5,
    label="Shared initial point",
)
x_max = max(
    int(zero_plot_df["interaction"].max()),
    int(tr_plot_df["cumulative_composite_queries"].max()),
)
ax.set_xlim(0, x_max)
ax.set_xlabel("Cumulative composite high-fidelity queries")
ax.set_ylabel("Best upper objective / Loss")
ax.set_title("Composite-query accounting")
ax.grid(alpha=0.35)
ax.legend()

fig.tight_layout()
plt.show()

output_path = Path("plots/Cold_start_query_accounting42.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)

fig.tight_layout()
fig.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
)
plt.show()

print(f"Saved figure to: {output_path.resolve()}")